<a href="https://colab.research.google.com/github/naufalsetiawan27/GA-WFC-PCG/blob/main/GA%2BWFC_Level_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random as rd
import numpy as np
import math
from collections import deque

In [ ]:
rng = np.random.default_rng()

# Gene

In [ ]:
class Gene():
  def __init__(self, len_h, len_x, dense, hazard):
    self.len_h = len_h
    self.len_x = len_x
    self.dense = dense
    self.hazard = hazard
    self.grid = None #fenotip

In [ ]:
len_x = rng.uniform(3, 7)
len_h = rng.uniform(1, 5)
dense = rng.uniform(0, 1)
hazard = rng.uniform(0,1)

print(len_x)
print(len_h)
print(dense)
print(hazard)

5.368474060669137
2.601052792783084
0.7663690857675175
0.9801988680643889


In [ ]:
gen_1 = Gene(len_h, len_x, dense, hazard)

In [ ]:
print(gen_1)

# Tile Config

In [ ]:
from enum import IntEnum

class Tile(IntEnum):
    EMPTY    = 0
    SOLID    = 1
    HAZARD   = 2

In [ ]:
ADJACENCY = {
    Tile.EMPTY : {
        "up"    : {Tile.EMPTY},
        "left"  : {Tile.EMPTY,Tile.SOLID,Tile.HAZARD},
        "right" : {Tile.EMPTY,Tile.SOLID,Tile.HAZARD},
        "down"  : {Tile.EMPTY,Tile.SOLID,Tile.HAZARD},
    },

    Tile.SOLID : {
        "up"    : {Tile.EMPTY,Tile.HAZARD,Tile.SOLID},
        "left"  : {Tile.EMPTY,Tile.HAZARD,Tile.SOLID},
        "right" : {Tile.EMPTY,Tile.HAZARD,Tile.SOLID},
        "down"  : {Tile.SOLID},
    },

    Tile.HAZARD : {
        "up"    : {Tile.EMPTY},
        "left"  : {Tile.EMPTY,Tile.SOLID,Tile.HAZARD},
        "right" : {Tile.EMPTY,Tile.SOLID,Tile.HAZARD},
        "down"  : {Tile.SOLID},
    },
}

BASE_WEIGHTS = {
    Tile.EMPTY  : 0.45,
    Tile.SOLID  : 0.45,
    Tile.HAZARD : 0.1,
}

DIRECTIONS = {
    "up":    (-1, 0),
    "down":  ( 1, 0),
    "left":  ( 0,-1),
    "right": ( 0, 1),
}

# WFC

In [ ]:
def get_weights(dense: float, hazard: float):
    weights = BASE_WEIGHTS.copy()
    weights[0] *= (1 - dense) # empty
    weights[1] *= dense # solid
    weights[2] *= hazard

    total = sum(weights.values())
    weights = {k: v / total for k, v in weights.items()}
    return weights

In [ ]:
def shannon_entropy(possible_tiles: set, dense: float, hazard: float) -> float:
    weights = get_weights(dense, hazard)
    total_weight = sum(weights[t] for t in possible_tiles if weights[t] > 0)
    entropy = 0.0
    for t in possible_tiles:
        w = weights[t]
        if w <= 0:
            continue
        p = w / total_weight
        entropy -= p * math.log(p)
    return entropy

In [ ]:
def weighted_random_choice(possible_tiles: set, dense: float, hazard: float) -> Tile:
    weights = get_weights(dense, hazard)
    tiles   = [t for t in possible_tiles if weights[t] > 0]
    # print(tiles)
    w  = [weights[t] for t in tiles]
    # print(w)
    selected_tiles = rd.choices(tiles, weights=w, k=1)
    return selected_tiles[0]

In [ ]:
def propagate(grid, isCollapsed, start_row, start_col, rows, cols) -> bool:
    queue = deque([(start_row, start_col)])

    while queue:
        row, col = queue.popleft()
        '''DIRECTIONS = {
              "up":    (-1, 0),
              "down":  ( 1, 0),
              "left":  ( 0,-1),
              "right": ( 0, 1),
            }'''

        # iterasi 4 arah
        for direction, (dr, dc) in DIRECTIONS.items():

            next_row, next_col = row + dr, col + dc

            if not (0 <= next_row < rows and 0 <= next_col < cols):
                continue

            if isCollapsed[next_row][next_col]:
                continue

            allowed = set()
            for tile in grid[row][col]:# grid[][] -> shape: {Tile.ITEM,...}
                allowed |= ADJACENCY[tile][direction] # shape: {Tile.ITEM,...}

            before = grid[next_row][next_col] # grid[][] -> shape: {Tile.ITEM,...}
            after = before & allowed # shape: {Tile.ITEM,...} intersect

            if len(after) == 0:
                return True # contradiction

            if after != before:
                grid[next_row][next_col] = after
                queue.append((next_row, next_col))

    return False

In [ ]:
def run_wfc(len_x, len_h, dense, hazard
            # size : list[int], enter_row : int, exit_row:int
            ) -> list[list[Tile]]:
    cols = round(len_x)
    rows = round(len_h)
    possible_tiles = set(Tile)

    # initialize rows*cols matric with copy of possible tiles,
    # shape: [EMPTY,SOLID,HAZARD] (at first the possible tiles is all type of tiles)
    grid = [[set(possible_tiles) for _ in range(cols)] for _ in range(rows)]
    isCollapsed = [[False] * cols for _ in range(rows)]

    for _ in range(rows * cols):
        min_entropy = float("inf")
        candidates = []

        # iterate each cell, searching for cells with minimum entropy score
        for r in range(rows):
            for c in range(cols):

                # if collapsed, continue
                if isCollapsed[r][c]:
                    continue

                # if no possible tiles, none
                if len(grid[r][c]) == 0:
                    return None

                # calculate entropy
                s = shannon_entropy(grid[r][c],dense, hazard)

                #
                if s < min_entropy:
                    min_entropy = s
                    candidates = [(r,c)]

                elif s == min_entropy:
                    candidates.append((r,c))

        if not candidates:
            break

        r, c = rd.choice(candidates)

        # collapsing cell
        chosen_tile =  weighted_random_choice(grid[r][c], dense, hazard)
        grid[r][c] = {chosen_tile}
        isCollapsed[r][c] = True

        # propagate
        if propagate(grid, isCollapsed, r, c, rows, cols):
            return None  # contradiction

    # finalize uncollapsed cells
    for r in range(rows):
        for c in range(cols):
            if not isCollapsed[r][c]:
                if len(grid[r][c]) == 0:
                    return None
                grid[r][c] = {weighted_random_choice(grid[r][c],dense, hazard)}

    result = []

    for r in range(rows):
        row = []
        for c in range(cols):
            value = next(iter(grid[r][c]))
            row.append(value)
        result.append(row)

    return result


In [ ]:
def generate_chunk(gen: Gene, max_retries: int = 10):
    dense, hazard = gen.dense, gen.hazard
    for attempt in range(max_retries):
        result = run_wfc(gen.len_x, gen.len_h, dense, hazard)
        if result is not None:
            gen.grid = result
            return gen
        print(f"  [WFC] Contradiction attempt {attempt + 1}, retrying...")
    raise RuntimeError(f"WFC failed after {max_retries} attempts.")

In [ ]:
def print_chunk(chunk: list):
    for row in chunk:
        print("[" + ",".join(str(int(t)) for t in row) + "],")

In [ ]:
def get_chunk(chunk) -> list:
    return [[int(t) for t in row] for row in chunk.grid]

In [ ]:
len_x = rng.uniform(3, 7)
len_x_1 = rng.uniform(3, 7)
len_h = rng.uniform(1, 5)
len_h_1 = rng.uniform(1, 5)
dense = rng.uniform(0, 1)
hazard = rng.uniform(0,1)

gen_2 = Gene(len_h, len_x, dense, hazard)
gen_1 = Gene(len_h_1, len_x_1, dense, hazard)
chunk_2 = generate_chunk(gen_2)
chunk_1 = generate_chunk(gen_1)
c = get_chunk(chunk_2)
d = get_chunk(chunk_1)
print(c)
print(d)
z = [c,d]
print([c, d])
print(gen_2.grid)
print(gen_1.grid)

[[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]]
[[0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 0, 1, 0]]
[[[0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], [[0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 0, 1, 0]]]
[[<Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>], [<Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>], [<Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>], [<Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>]]
[[<Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.EMPTY: 0>], [<Tile.EMPTY: 0>, <Tile.EMPTY: 0>, <Tile.SOLID: 1>, <Tile.SOLID: 1>, <Tile.EMPTY: 0>, <Tile.SOLID: 1>, <Tile.EMPTY: 0>]]


# Evaluate

In [ ]:
def embed(c, screen_height = 10):
  screen = []
  for _ in c:
    # print("array", _)
    y = len(_[0])
    # print("y", y)
    x = len(_)
    # print("x", x)
    plc = np.zeros((screen_height, y))
    # screen = [[0 for rows in range(y)] for cols in range(screen_height)]
    # print(plc)
    plc[-x:] = _
    # print(plc)
    screen.append(plc)

  return np.concatenate(screen, axis=-1).tolist()

print(embed(z))


[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [1.0, 1.0, 0.0, 1.0, 1.0, 0.0, 1.0], [1.0, 1.0, 0.0, 1.0, 1.0, 2.0, 1.0], [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]]


In [ ]:
genes = []
for _ in range(5):
  len_x = rng.uniform(3, 7)
  len_h = rng.uniform(1, 5)
  dense = rng.uniform(0, 1)
  hazard = rng.uniform(0,1)

  gene = Gene(len_h, len_x, dense, hazard)
  genes.append(gene)
print(genes)

chunks = []
for gen in genes:
  # print(gen.grid)
  chunk = generate_chunk(gen)
  # print(chunk)
  # print(gen.grid)
  # print(chunk)
  chunk = get_chunk(chunk)
  chunks.append(chunk)

print(chunks)

X = embed(chunks)
print(X)


[<__main__.Gene object at 0x7d67e3fde2a0>, <__main__.Gene object at 0x7d67e3fdeb70>, <__main__.Gene object at 0x7d67e3fde2d0>, <__main__.Gene object at 0x7d67da642450>, <__main__.Gene object at 0x7d67da407ec0>]
[[[1, 1, 0, 1, 1, 0], [1, 1, 0, 1, 1, 0]], [[1, 1, 1, 0, 1], [1, 1, 1, 1, 1]], [[1, 2, 1, 1, 1], [1, 1, 1, 1, 1], [1, 1, 1, 1, 1]], [[0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 1], [0, 0, 1, 0, 2, 1]], [[0, 0, 1, 0, 0, 0], [0, 0, 1, 0, 0, 0], [0, 0, 1, 0, 0, 0], [0, 0, 1, 2, 2, 0]]]
[[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

In [ ]:
X = np.array(X)
print(X)

[[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 2. 1. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0.
  1. 0. 0. 0.]
 [1. 1. 0. 1. 1. 0. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 1. 0. 0.
  1. 0. 0. 0.]
 [1. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 0. 2. 1. 0. 0.
  1. 2. 2. 0.]]


In [ ]:
def get_platform_heights(grid: list[list]) -> np.ndarray:
    SOLID = 1
    g = np.array(grid)
    heights = []
    for col in range(g.shape[1]):
        col_data = g[:, col]
        filled = np.where(col_data == SOLID)[0]  # hanya solid
        if len(filled) == 0:
            heights.append(np.nan)  # gap atau kolom isi hazard semua
        else:
            heights.append(filled[0])
    return np.array(heights)

print(get_platform_heights(X))

[ 8.  8. nan  8.  8. nan  8.  8.  8.  9.  8.  7.  8.  7.  7.  7. nan nan
  9. nan nan  7. nan nan  6. nan nan nan]


In [ ]:
import numpy as np
from dataclasses import dataclass
from typing import Callable

@dataclass
class DifficultyConfig:
    window_width: int = 5          # lebar window dalam kolom
    stride: int = 2                # step per slide
    w_gap: float = 0.4             # bobot gap difficulty
    w_hazard: float = 0.3          # bobot hazard density
    w_smooth: float = 0.3          # bobot terrain roughness
    target_fn: Callable = None     # fungsi target kurva difficulty

In [ ]:
config = DifficultyConfig()

In [ ]:
# hitung diff over time (per window)
def window_diff(
    grid_slice: np.ndarray,   # shape: (screen_h, window_w)
    w_gap, w_hazard, w_smooth) -> float:

    chunk = np.array(grid_slice)
    cols = chunk.shape[1]

    # === GAP SCORE ===
    SOLID = 1
    cols_with_platform = [np.any(chunk[:, col] == SOLID) for col in range(cols)] # -> [bool]
    # print(cols_with_platform)

    gaps = []
    consec_gap = 0 # gap berurutan
    for has_platform in cols_with_platform:
        if not has_platform: # jika tidak ada platform, tambah gap
            consec_gap += 1
        else: # jika ada platform
            if consec_gap > 0: # jika sebelumnya adalah gap
                gaps.append(consec_gap)
            consec_gap = 0
    if consec_gap > 0:
        gaps.append(consec_gap)

    # if gap == panjang window -> diff = 1
    # normalize
    gap_score = sum(r**2 for r in gaps) / (cols**2) if gaps else 0.0
    # print(gaps)
    # print(f'gap_score = {gap_score}')

    # === HAZARD SCORE ===
    HAZARD = 2
    n_walkable = sum(1 for col in range(cols) if np.any(chunk[:, col] > 0))
    n_hazard = np.sum(chunk == HAZARD)
    hazard_score = n_hazard / max(n_walkable, 1)
    # print(f'hazard_score = {hazard_score}')

    # === SMOOTHNESS SCORE ===
    heights = get_platform_heights(chunk.tolist())
    # print(heights)
    valid_h = heights[~np.isnan(heights)] # Nan == gap
    if len(valid_h) > 1:
        diffs = np.diff(valid_h)
        smooth_score = np.var(diffs) / (chunk.shape[0]**2)  # normalize by max rows
    else:
        smooth_score = 0.0
    # print(f'smooth_score = {smooth_score}')
    total_diff = ((w_gap * gap_score) + (w_hazard * hazard_score) + (w_smooth * smooth_score))

    return total_diff


In [ ]:
def difficulty_curve(grid: list[list], config: DifficultyConfig) -> np.ndarray:
    g = np.array(grid)
    total_width = g.shape[1]
    scores = []

    for start in range(0, total_width - config.window_width + 1, config.stride):
        end = start + config.window_width
        window = g[:, start:end]
        scores.append(window_diff(grid_slice = window.tolist(),
                                  w_gap = config.w_gap,
                                  w_hazard = config.w_hazard,
                                  w_smooth = config.w_smooth))

    return np.array(scores)

In [ ]:
import matplotlib.pyplot as plt

In [1]:
curve = difficulty_curve(X, config)

# normalize
if curve.max() > curve.min():
    curve_norm = (curve - curve.min()) / (curve.max() - curve.min())
else:
    curve_norm = np.zeros_like(curve)

x = np.linspace(0, 1, len(curve_norm))

target_y = x

plt.figure(figsize=(10, 4))
plt.plot(x, curve_norm, marker='o', linewidth=2, label='difficulty curve')
plt.plot(x, target_y, '--', linewidth=2, label='target (linear)', alpha=0.7)
plt.xlabel('progress (normalized)')
plt.ylabel('difficulty (normalized)')
plt.ylim(-0.05, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'difficulty_curve' is not defined

# GA

In [ ]:
def init_population(len_chunk : float, # chromosome length
                    no_grid : bool = True,
                    n_pop : float = 30, # number of solutions
                    ) -> list[list]:
  rng = np.random.default_rng()
  levels = []

  for i in range(n_pop):
    level = []
    for c in range(len_chunk):
      len_x = rng.uniform(3, 7)
      len_h = rng.uniform(1, 5)
      dense = rng.uniform(0, 1)
      hazard = rng.uniform(0,1)

      chunk = Gene(len_h, len_x, dense, hazard)

      # generate grid
      if not no_grid:
        generate_chunk(chunk)

      level.append(chunk)
    levels.append(level)

  return levels

In [ ]:
solution  = init_population(10)
solution_2  = init_population(10, False)

In [ ]:
print(solution[0][0].grid)
print(solution_2[0][0].grid)

None
[[<Tile.SOLID: 1>, <Tile.EMPTY: 0>, <Tile.SOLID: 1>], [<Tile.SOLID: 1>, <Tile.EMPTY: 0>, <Tile.SOLID: 1>], [<Tile.SOLID: 1>, <Tile.HAZARD: 2>, <Tile.SOLID: 1>]]
